In [3]:
import pandas as pd
import math
import matplotlib.pyplot as plt
import numpy as np
import importlib
import functions as f
from scintkit.preprocessing.format import temp_formating
from scintkit.services.phase_detrend import detect_sampling_rate
from pathlib import Path
import time
import CONFIG as cf

importlib.reload(f)
importlib.reload(cf)


<module 'CONFIG' from 'c:\\Users\\irees\\Downloads\\Summer_learning\\research26\\scintkit_summer\\src\\scintkit\\space_receiver_processing\\CONFIG.py'>

In [ ]:

start = time.time()

f.initialize_log(cf.log_file) #prepare the log file

# ============================================================
# 1. FILE ORGANIZATION
# ============================================================

print(f'started at {start- start}')
#create file organization code:

file_org_start = time.time()
files = f.find_files(cf.input_directory)

receiverA_files, receiverB_files = f.org_receivers(files, cf.r_latitude, cf.r_longitude, cf.lat_tol, cf.lon_tol)

    #finally have all the receiver data organized into 2 seperate files 

file_org_time = time.time() - file_org_start

# ----- Verification -----
print(f"Found {len(files)} total files")
print(f"Receiver A: {len(receiverA_files)} files")
print(f"Receiver B: {len(receiverB_files)} files")

if len(receiverA_files) == 0:
    raise ValueError("No files were assigned to Receiver A.")

if len(receiverB_files) == 0:
    raise ValueError("No files were assigned to Receiver B.")

print(f"File organization time: {file_org_time:.3f} seconds")


# ============================================================
# 2. FILE PAIRING
# ============================================================
pair_start = time.time()

paired_files = f.pair_receiver_files(receiverA_files, receiverB_files, cf)
pair_time = time.time() - pair_start

print(f"Created {len(paired_files)} valid file pairs")
print(f"File pairing time: {pair_time:.3f} seconds")
# ============================================================
# TOTAL TIMERS
# ============================================================
total_read_time = 0
total_preprocess_time = 0
total_merge_time = 0
total_s4_time = 0
total_corr_time = 0

# ============================================================
# PROCESS FILE PAIRS
# ============================================================

i = 0
all_scint = []
for fileA, fileB in paired_files:
    i += 1
    print(f"\nProcessing:")
    print(fileA)
    print(fileB)

    # --------------------------------------------------------
    # 3. READ FILES
    # --------------------------------------------------------

    read_start = time.time()
    dfa = pd.read_parquet(fileA)
    dfb = pd.read_parquet(fileB)

    rAloc = f.extract_coord(fileA)
    rBloc = f.extract_coord(fileB)

    read_time = time.time() - read_start
    total_read_time += read_time

    print(f"Read files: {read_time:.3f} seconds")


    # --------------------------------------------------------
    # 4. PREPROCESSING
    # --------------------------------------------------------

    preprocess_start = time.time()

    # filter dfs to contain certain elevation
    dfa = dfa[dfa['elev'] > cf.elevation_filter].copy()
    dfb = dfb[dfb['elev'] > cf.elevation_filter].copy()


    # individual sampling rates
    dfa = temp_formating(dfa)
    dfb = temp_formating(dfb)


    samp_ra = detect_sampling_rate(dfa)
    dt = 1 / samp_ra


    preprocess_time = time.time() - preprocess_start
    total_preprocess_time += preprocess_time

    print(f"Preprocessing: {preprocess_time:.3f} seconds")


    # --------------------------------------------------------
    # 5. MERGE
    # --------------------------------------------------------

    merge_start = time.time()

    #add s4 filtering here in method 2
    # merge 2 receiver dfs
    merged = dfa.merge(dfb, on=["datetime", "svid", "cons"], suffixes=("_A", "_B"))
    merged['snr_diff'] = abs(merged['snr1_A'] - merged['snr1_B'])

    merge_time = time.time() - merge_start
    total_merge_time += merge_time

    print(f"Merge: {merge_time:.3f} seconds")


    # --------------------------------------------------------
    # 6. S4 + CROSS CORRELATION
    # --------------------------------------------------------

    thresh = cf.thresh  # threshold for s4 scintillation measurement

    #### start cross corr file creation

    rstart = time.time()

    scint = []

    sat_groups = merged.groupby(['svid', 'cons'])

    for (svid, cons), sat_group in sat_groups:


        # group the data into different satellites
        sat_group = f.datetime_to_seconds(sat_group)

        min_groups = sat_group.groupby(sat_group['datetime'].dt.floor('min'))

        # with the chosen satellite for this iteration
        # find s4 and then determine scintillation
        for min, group in min_groups:

            # -----------------------------------------------
            # S4 TIMER
            # -----------------------------------------------

            s4_start = time.time()

            group = f.handle_nan(group, cf.nan_method, sig_columns = ['snr1_A', 'snr1_B'])
            #add nan processing
            if len(group) < 10: #makes sure there is enough samples to actually process data
                continue

            
            #snr 1
            sig_1_lina = f.db2lin(group['snr1_A'])
            s4_1_a = np.std(sig_1_lina) / np.mean(sig_1_lina)

            sig_1_linb = f.db2lin(group['snr1_B'])
            s4_1_b = np.std(sig_1_linb) / np.mean(sig_1_linb)

            #snr 2 (snr 3 is all 0 or nan)
            sig_2_lina = f.db2lin(group['snr2_A'])
            s4_2_a = np.std(sig_2_lina) / np.mean(sig_2_lina)

            sig_2_linb = f.db2lin(group['snr2_B'])
            s4_2_b = np.std(sig_2_linb) / np.mean(sig_2_linb)


            s4_time = time.time() - s4_start
            total_s4_time += s4_time

            # -----------------------------------------------
            # CROSS CORRELATION
            # -----------------------------------------------

            if s4_1_a > thresh or s4_1_b > thresh:

                corr_start = time.time()
                # check threshold of scintillation and compute correlation and run auto correlation
                correlation, lag_b, cor_norm, lag_norm = f.cross_correlation(group["snr1_A"], group["snr1_B"])
                autoA_max, autoA_lagb, autoA_cor, autoA_lags = f.cross_correlation(group['snr1_A'], group['snr1_A'])
                autoB_max, autoB_lagb, autoB_cor, autoB_lags = f.cross_correlation(group['snr1_B'], group['snr1_B'])

                time_delay = lag_b * dt

                scint.append({
                    'minute': min,
                    'prn' : group['prn_B'].iloc[0],
                    's4_1_a': s4_1_a,
                    's4_1_b': s4_1_b,
                    's4_2_a': s4_2_a,
                    's4_2_b': s4_2_b,
                    

                    #adding elev and azim
                    'elev' : group['elev_A'].mean(),
                    'azim' : group['azim_A'].mean(),
                    
                    #adding location, using the location at the start of each minute not the mean, can be changed
                    'r_a' : rAloc,
                    'r_b' : rBloc,

                    'auto_cor_a' : autoA_cor,
                    'auto_cor_a_max' : autoA_max,
                    'auto_cor_b' : autoB_cor,
                    'auto_cor_b_max' : autoB_max,


                    'corr_norm': cor_norm,
                    'lag_norm': lag_norm,
                    'max_corr': correlation,
                    'best_lag': lag_b,
                    'time_delay': time_delay
                })
                corr_time = time.time() - corr_start
                total_corr_time += corr_time
    f.log_processed_pair(fileA, fileB, cf.log_file)
    all_scint.extend(scint)
    
    print(f'processed {i} files')
    print(f"time to process file pair: {time.time() - rstart:.3f} seconds")

# ============================================================
# 7. DISTANCE CALCULATION
# ============================================================

# create dataframe storing all scintillation events with distance

cross_cor = pd.DataFrame(all_scint)

print(f"Processed {len(cross_cor)} scintillation events")
print(f"Runtime: {time.time() - start:.3f} seconds")

dist_start = time.time()
# add distance
cross_cor['distance (km)'] = cross_cor.apply(lambda row: f.calc_dist(row['r_a'], row['r_b']), axis = 1)

distance_time = time.time() - dist_start

# ============================================================
# PRINT TOTAL TIMES
# ============================================================

total_runtime = time.time() - start

print("\n========================================")
print("RUNTIME BREAKDOWN")
print("========================================")

print(f"File organization:    {file_org_time:.3f} sec")
print(f"Reading files:        {total_read_time:.3f} sec")
print(f"File pairing:         {pair_time:.3f} sec")
print(f"Preprocessing:        {total_preprocess_time:.3f} sec")
print(f"Merging:              {total_merge_time:.3f} sec")
print(f"S4 calculation:       {total_s4_time:.3f} sec")
print(f"Cross-correlation:    {total_corr_time:.3f} sec")
print(f"Distance calculation: {distance_time:.3f} sec")

print("----------------------------------------")
print(f"TOTAL RUNTIME:        {total_runtime:.3f} sec")
print("========================================")

#use extend to aviod appending lists


started at 0.0
Found 98 total files
Receiver A: 49 files
Receiver B: 49 files
File organization time: 0.006 seconds
Skipping scintpi3_20250325_1552_359062.2500W_72122.5234S_v326d_lvl0.pq
Skipping scintpi3_20250326_1552_359062.2188W_72122.3750S_v326d_lvl0.pq
Skipping scintpi3_20250327_1552_359062.1562W_72122.2031S_v326d_lvl0.pq
Skipping scintpi3_20250328_1552_359062.4375W_72122.2578S_v326d_lvl0.pq
Skipping scintpi3_20250329_1552_359062.2188W_72122.2188S_v326d_lvl0.pq
Skipping scintpi3_20250330_1552_359062.3438W_72121.8984S_v326d_lvl0.pq
Skipping scintpi3_20250331_1552_359062.3125W_72122.4062S_v326d_lvl0.pq
Created 42 valid file pairs
File pairing time: 0.002 seconds

Processing:
C:\Users\irees\Downloads\Summer_learning\research26\1_month_test-selected_lvl0_25-31\scintpi3_20250325_0000_359062.5312W_72122.7500S_v326d_lvl0.pq
C:\Users\irees\Downloads\Summer_learning\research26\1_month_test-selected_lvl0_25-31\scintpi3_20250325_0000_359072.5625W_72127.2500S_v326d_lvl0.pq
Read files: 1.784 s

In [5]:
#change to file for each day
file_time = time.time()

base_name = Path(cf.cross_correlation_file).stem


output_folder = Path(cf.output_folder)
output_folder.mkdir(parents=True, exist_ok=True)


for day, day_df in cross_cor.groupby(cross_cor['minute'].dt.date):
    day_str = pd.Timestamp(day).strftime('%Y%m%d')

    output_path = output_folder / (f'{base_name}_{day_str}'
                                   f'_A_{cf.r_latitude:.5f}_{cf.r_longitude:.5f}.pq')
    day_df.to_parquet(output_path, index = False)

    print(f'Saved to {output_path.name}')

print(f'time to save files: {time.time - file_time} seconds')

print(f"Total Time: {time.time() - start:.3f} seconds")



Saved to _corrs_20250325_A_7.21224_35.90622.pq
Saved to _corrs_20250326_A_7.21224_35.90622.pq
Saved to _corrs_20250327_A_7.21224_35.90622.pq
Saved to _corrs_20250328_A_7.21224_35.90622.pq
Saved to _corrs_20250329_A_7.21224_35.90622.pq
Saved to _corrs_20250330_A_7.21224_35.90622.pq
Saved to _corrs_20250331_A_7.21224_35.90622.pq
Saved to _corrs_20250401_A_7.21224_35.90622.pq


TypeError: unsupported operand type(s) for -: 'builtin_function_or_method' and 'float'

In [ ]:


display(cross_cor)

cross_cor['velocity (km/s)'] = cross_cor['distance (km)']/cross_cor['time_delay']
sat = cross_cor[cross_cor['prn'] == 'G05'] 
display(sat)



,minute,prn,s4_1_a,s4_1_b,s4_2_a,s4_2_b,elev,azim,r_a,r_b,auto_cor_a,auto_cor_a_max,auto_cor_b,auto_cor_b_max,corr_norm,lag_norm,max_corr,best_lag,time_delay,distance (km)
0,2025-03-25 01:30:00,G05,0.061652,0.202286,0.158117,0.466766,21.000000,4.0000,"(7.212275, 35.90625312, 0)","(7.212725, 35.90725625, 0)","[0.0005304299274148729, 0.0010608598548297458,...",1.0,"[-0.0009505399927815897, -0.001901079985563179...",1.0,"[-0.0003165936009922179, -0.000633187201984435...","[-198, -197, -196, -195, -194, -193, -192, -19...",0.367872,-100,-5.00,0.121447
1,2025-03-25 01:31:00,G05,0.115363,0.340965,0.115340,0.212193,21.000000,4.0000,"(7.212275, 35.90625312, 0)","(7.212725, 35.90725625, 0)","[-0.004166666666666659, -0.008333333333333318,...",1.0,"[-0.0013757375749236132, -2.2739464048329487e-...",1.0,"[-0.001554370629912459, -0.003108741259824918,...","[-239, -238, -237, -236, -235, -234, -233, -23...",0.529632,3,0.15,0.121447
2,2025-03-25 01:32:00,G05,0.186208,0.228114,0.111654,0.130709,21.000000,4.9125,"(7.212275, 35.90625312, 0)","(7.212725, 35.90725625, 0)","[0.0051087589116833925, 0.010217517823366785, ...",1.0,"[-0.0010928961748633865, -0.00400728597449908,...",1.0,"[-0.0019293078332013245, -0.003858615666402649...","[-239, -238, -237, -236, -235, -234, -233, -23...",0.417404,-23,-1.15,0.121447
3,2025-03-25 01:33:00,G05,0.186873,0.277663,0.094498,0.247955,21.000000,5.0000,"(7.212275, 35.90625312, 0)","(7.212725, 35.90725625, 0)","[-0.0006427732406327275, -0.001285546481265455...",1.0,"[-0.005933158524243261, -0.011866317048486522,...",1.0,"[0.0008062830598903445, 0.001612566119780689, ...","[-239, -238, -237, -236, -235, -234, -233, -23...",0.618548,-34,-1.70,0.121447
4,2025-03-25 01:34:00,G05,0.158838,0.300268,0.112084,0.141074,21.807512,5.0000,"(7.212275, 35.90625312, 0)","(7.212725, 35.90725625, 0)","[-0.0014323367829467523, -0.002864673565893504...",1.0,"[-0.0007699688458759287, -0.000652170193387221...",1.0,"[-0.0016600937789977776, 0.001213145453882986,...","[-212, -211, -210, -209, -208, -207, -206, -20...",0.348969,-32,-1.60,0.121447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20670,2025-03-31 23:52:00,S136,0.422124,0.440102,NaN,NaN,42.000000,82.0000,"(7.2122398400000005, 35.90622188, 0)","(7.2127007800000005, 35.9072625, 0)","[0.001649862465593307, 0.003299724931186614, 0...",1.0,"[0.0003217526701214015, 0.000643505340242803, ...",1.0,"[0.0009559413772733022, 0.0019118827545466044,...","[-719, -718, -717, -716, -715, -714, -713, -71...",0.931445,10,0.50,0.125718
20671,2025-03-31 23:53:00,S136,0.289731,0.295387,NaN,NaN,42.000000,82.0000,"(7.2122398400000005, 35.90622188, 0)","(7.2127007800000005, 35.9072625, 0)","[0.0006318090275732183, 0.0012636180551464367,...",1.0,"[0.0016501061455762578, 0.0033002122911525156,...",1.0,"[0.0012344410154645868, 0.0024688820309291737,...","[-719, -718, -717, -716, -715, -714, -713, -71...",0.906980,10,0.50,0.125718
20672,2025-03-31 23:54:00,S136,0.367669,0.355091,NaN,NaN,42.000000,82.0000,"(7.2122398400000005, 35.90622188, 0)","(7.2127007800000005, 35.9072625, 0)","[0.00014106988204174687, 0.0002821397640834937...",1.0,"[0.0005275770350296119, 0.0010551540700592238,...",1.0,"[0.0002728098790358625, 0.000545619758071725, ...","[-719, -718, -717, -716, -715, -714, -713, -71...",0.924853,10,0.50,0.125718
20673,2025-03-31 23:55:00,S136,0.268016,0.276612,NaN,NaN,42.000000,82.0000,"(7.2122398400000005, 35.90622188, 0)","(7.2127007800000005, 35.9072625, 0)","[-0.0008340305375444446, -0.001668061075088889...",1.0,"[-0.0018988742946256841, -0.003797748589251368...",1.0,"[-0.0011409889992497026, -0.002281977998499405...","[-719, -718, -717, -716, -715, -714, -713, -71...",0.879279,9,0.45,0.125718


,minute,prn,s4_1_a,s4_1_b,s4_2_a,s4_2_b,elev,azim,r_a,r_b,...,auto_cor_a_max,auto_cor_b,auto_cor_b_max,corr_norm,lag_norm,max_corr,best_lag,time_delay,distance (km),velocity (km/s)
0,2025-03-25 01:30:00,G05,0.061652,0.202286,0.158117,0.466766,21.000000,4.000000,"(7.212275, 35.90625312, 0)","(7.212725, 35.90725625, 0)",...,1.0,"[-0.0009505399927815897, -0.001901079985563179...",1.0,"[-0.0003165936009922179, -0.000633187201984435...","[-198, -197, -196, -195, -194, -193, -192, -19...",0.367872,-100,-5.00,0.121447,-0.024289
1,2025-03-25 01:31:00,G05,0.115363,0.340965,0.115340,0.212193,21.000000,4.000000,"(7.212275, 35.90625312, 0)","(7.212725, 35.90725625, 0)",...,1.0,"[-0.0013757375749236132, -2.2739464048329487e-...",1.0,"[-0.001554370629912459, -0.003108741259824918,...","[-239, -238, -237, -236, -235, -234, -233, -23...",0.529632,3,0.15,0.121447,0.809650
2,2025-03-25 01:32:00,G05,0.186208,0.228114,0.111654,0.130709,21.000000,4.912500,"(7.212275, 35.90625312, 0)","(7.212725, 35.90725625, 0)",...,1.0,"[-0.0010928961748633865, -0.00400728597449908,...",1.0,"[-0.0019293078332013245, -0.003858615666402649...","[-239, -238, -237, -236, -235, -234, -233, -23...",0.417404,-23,-1.15,0.121447,-0.105607
3,2025-03-25 01:33:00,G05,0.186873,0.277663,0.094498,0.247955,21.000000,5.000000,"(7.212275, 35.90625312, 0)","(7.212725, 35.90725625, 0)",...,1.0,"[-0.005933158524243261, -0.011866317048486522,...",1.0,"[0.0008062830598903445, 0.001612566119780689, ...","[-239, -238, -237, -236, -235, -234, -233, -23...",0.618548,-34,-1.70,0.121447,-0.071440
4,2025-03-25 01:34:00,G05,0.158838,0.300268,0.112084,0.141074,21.807512,5.000000,"(7.212275, 35.90625312, 0)","(7.212725, 35.90725625, 0)",...,1.0,"[-0.0007699688458759287, -0.000652170193387221...",1.0,"[-0.0016600937789977776, 0.001213145453882986,...","[-212, -211, -210, -209, -208, -207, -206, -20...",0.348969,-32,-1.60,0.121447,-0.075905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16706,2025-03-31 01:06:00,G05,0.276722,0.278440,0.418102,0.483946,21.000000,4.000000,"(7.21218125, 35.90633125, 0)","(7.21261094, 35.9073375, 0)",...,1.0,"[0.005380412521434885, 0.01076082504286977, 0....",1.0,"[0.00988386486412168, 0.016573817064728057, 0....","[-147, -146, -145, -144, -143, -142, -141, -14...",0.706447,1,0.05,0.120851,2.417013
16707,2025-03-31 01:07:00,G05,0.352526,0.565176,0.604677,0.576408,21.000000,4.000000,"(7.21218125, 35.90633125, 0)","(7.21261094, 35.9073375, 0)",...,1.0,"[-0.0036873256138771973, -0.002679065404510591...",1.0,"[0.008687471111682344, 0.015661718668061677, 0...","[-239, -238, -237, -236, -235, -234, -233, -23...",0.607166,1,0.05,0.120851,2.417013
16708,2025-03-31 01:08:00,G05,0.393766,0.465342,0.438590,0.550753,21.000000,4.666667,"(7.21218125, 35.90633125, 0)","(7.21261094, 35.9073375, 0)",...,1.0,"[-0.012843060246327585, -0.02568612049265517, ...",1.0,"[-0.01233107519904492, -0.027770824818017126, ...","[-239, -238, -237, -236, -235, -234, -233, -23...",0.707040,2,0.10,0.120851,1.208507
16709,2025-03-31 01:09:00,G05,0.522749,0.544568,0.596596,0.596880,21.000000,5.000000,"(7.21218125, 35.90633125, 0)","(7.21261094, 35.9073375, 0)",...,1.0,"[0.0019995204414326798, 0.0005655209329304741,...",1.0,"[0.0011852632060719232, -0.0002828560827032788...","[-169, -168, -167, -166, -165, -164, -163, -16...",0.688857,1,0.05,0.120851,2.417013


In [ ]:
1/0


plt.plot(sat["minute"], sat["s4A"],'.-', label="Receiver A")
plt.plot(sat["minute"], sat["s4B"],'.-', label="Receiver B")
# plt.xlim(
#        pd.Timestamp("2022-10-04 23:00:00"),
#       pd.Timestamp("2022-10-04 23:01:00")
#     )
plt.legend()
plt.xlabel('Datetime')
plt.ylabel('s4')
plt.title('Reciever A and B s4 vs Datetime')
plt.xticks(rotation=45)
plt.show()



event = sat.iloc[40]
display(event)

plt.plot(event['lag_norm'], event['corr_norm'], label = 'Cross_cor')
plt.plot(event['lag_norm'], event['auto_cor_A'], label = 'Auto_cor_A')
plt.plot(event['lag_norm'], event['auto_cor_B'], label = 'Auto_cor_B')

text = (f"Time Delay = {event['time_delay']:.3f} s\n" f"Velocity = {event['velocity (km/s)']:.2f} km/s")

plt.text(
    0.02, 0.98,
    text,
    transform=plt.gca().transAxes,
    verticalalignment='top',
    bbox=dict(facecolor='white', alpha=0.8)
)


ZeroDivisionError: division by zero